# 07 · Huấn luyện PRDetect V4 trên HC3

Bật GPU và Internet rồi gắn bộ mã V4 vào Kaggle Input. Mỗi lượt huấn luyện V3 và A1 cùng seed. ZIP chứa EMA, A1 và manifest. Checkpoint RAW cho ablation nằm cùng thư mục kết quả. Ô cuối chấm HC3 sạch, GPT-3.5 sạch và HC3 Replace 25%.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from zipfile import ZipFile
from IPython.display import FileLink, display

WORK = Path('/kaggle/working/prdetect_v4')
REPO = WORK / 'PRDetect-Research'
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=True)
# The datasets and legacy model code used by the five original runs.
if not (REPO / 'evaluate_v3_quang.py').is_file():
    REPO.mkdir(exist_ok=True)
    subprocess.run(['git', 'init', '-q'], cwd=REPO, check=True)
    subprocess.run(['git', 'fetch', '--depth', '1', 'https://github.com/tphan1420/PRDetect-Research.git',
                    '9c8846d5198164124cfb19902fe65fb80e7387c5'], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO, check=True)

files = ('v4.py', 'v3_a1_fusion.py', 'train_v4.py', 'train_v3_dual_best.py')
archives = list(INPUT.rglob('prdetect_v4_code.zip'))
folders = [p.parent for p in INPUT.rglob('train_v4.py') if all((p.parent / f).is_file() for f in files)]
if len(archives) + len(folders) == 1:
    if archives:
        with ZipFile(archives[0]) as z:
            for name in files:
                (REPO / name).write_bytes(z.read(name))
    else:
        for name in files:
            shutil.copyfile(folders[0] / name, REPO / name)
elif archives or folders or not all((REPO / f).is_file() for f in files):
    raise ValueError('Gắn đúng một bản prdetect_v4_code.zip hoặc thư mục đã giải nén vào Kaggle Input.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-v3-eval.txt'),
                'torch-geometric'], check=True)
import spacy
if not spacy.util.is_package('en_core_web_sm'):
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings'
print(torch.cuda.get_device_name(0))

In [ ]:
SEED = 2026
V3_EPOCHS = 10
A1_EPOCHS = 10
BATCH_SIZE = 1
MICRO_BATCH_SIZE = 1
VAL_BATCH_SIZE = 64
A1_BATCH = 4
A1_LR = 0.0001
CHECKPOINT_DIR = WORK / 'checkpoints'
FEATURE_CACHE = WORK / 'feature_cache'

In [ ]:
subprocess.run([sys.executable, 'train_v4.py', '--seed', str(SEED),
                '--output-dir', str(CHECKPOINT_DIR), '--feature-cache', str(FEATURE_CACHE),
                '--v3-epochs', str(V3_EPOCHS), '--a1-epochs', str(A1_EPOCHS),
                '--batch-size', str(BATCH_SIZE), '--micro-batch-size', str(MICRO_BATCH_SIZE),
                '--val-batch-size', str(VAL_BATCH_SIZE), '--a1-batch', str(A1_BATCH), '--a1-lr', str(A1_LR)],
               cwd=REPO, check=True)
archive = CHECKPOINT_DIR / f'seed{SEED}' / f'v4_hc3_seed{SEED}.zip'
print(archive)
display(FileLink(str(archive)))

In [ ]:
OUT = WORK / f'three_tests_seed{SEED}'
subprocess.run([sys.executable, 'v4.py', 'evaluate', '--suite', 'three',
                '--checkpoint-dir', str(CHECKPOINT_DIR), '--seeds', str(SEED), '--batch-size', '1',
                '--device', 'cuda', '--output-dir', str(OUT)], cwd=REPO, check=True)
import pandas as pd
display(pd.read_csv(OUT / 'results_by_seed.csv'))
archive = shutil.make_archive(str(OUT), 'zip', OUT.parent, OUT.name)
display(FileLink(archive))